In [7]:
import os
import requests
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

url = os.environ["N8N_WEBHOOK_URL"]
data = {"location": "Stanley", "max_gust": 70}

r = requests.post(url, json=data)
print(r.status_code)
print(r.text)

200
{"message":"Workflow was started"}


In [18]:
from sqlalchemy import create_engine, text

engine = create_engine(
    "mssql+pyodbc://localhost/Alex_weather?trusted_connection=yes"
    "&driver=ODBC+Driver+17+for+SQL+Server&TrustServerCertificate=yes"
)

query = """
select top 1
    l.Location_Name,
    d.Full_Date,
    r.Max_Gust
from gold.Fact_Risk r
join gold.dim_location1 l on l.Location_SK = r.Location_SK
join gold.Dim_Date d      on d.Date_SK = r.Date_SK
where r.Risk_Level = 'High'
order by r.Max_Gust desc
"""

with engine.connect() as conn:
    row = conn.execute(text(query)).first()

if row is None:
    print("No High risk days")
else:
    url = os.environ["N8N_WEBHOOK_URL"]
    data = {
        "location": row.Location_Name,
        "date": str(row.Full_Date),
        "max_gust": float(row.Max_Gust),
    }
    r = requests.post(url, json=data)
    print(r.status_code, data)

200 {'location': 'Stanley', 'date': '2026-01-09', 'max_gust': 91.1}
